# Step 6 · From watching to nudging

**Pain–Comfort Lab · Looking inside a language model** · ⏱️ about 35 minutes · ⚡ GPU recommended

> ### 🤔 Wonder
> So far we've only *watched*. If we **push** the model's numbers along our pain → comfort arrow while it's writing, does it start describing sensations differently?

Our ruler shows that the direction **lines up** with pain-versus-comfort labels, on sentences that a simple word-counter can also sort. But lining up isn't the same as **mattering**. Maybe the model never uses that direction when deciding what to write.

To find out, we **experiment**: add our direction into the model's numbers while it runs, and see whether its next words change. Change one thing, keep everything else the same, watch the outcome.

**In this step you will:**

1. Turn the "clip-on sensor" hook into a hook that **changes** what flows past.
2. Check, by counting, that every nudge is removed afterwards.
3. Compare generated text under five conditions, including a random-arrow control.
4. Measure the effect with numbers instead of just reading text.

## Setup and catch-up

In [ ]:
# 🔧 Setup: just run this cell. It works in Google Colab and in Jupyter on your own computer.
# It makes the lab's helper library (the `pclab` folder) and the sentence files available.
import subprocess, sys
from pathlib import Path
from importlib.metadata import version
from packaging.version import Version

REPO_URL = 'https://github.com/23menon23/pain-comfort-lab'   # the lab's home on GitHub

if Path('../pclab').exists():                    # you are inside a downloaded copy of the repository
    LAB_ROOT = Path('..').resolve()
else:                                            # Google Colab: download the lab once per session
    LAB_ROOT = Path('pain-comfort-lab').resolve()
    if not LAB_ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(LAB_ROOT)], check=True)
sys.path.insert(0, str(LAB_ROOT))

if Version(version('transformers')) < Version('4.57'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.57'], check=True)
    print('⚠️ Transformers was updated. Choose Runtime → Restart session, then run this cell again.')
else:
    import pclab
    from pclab import look, show_source
    print('✅ Ready. pclab', pclab.__version__, '| Transformers', version('transformers'))

In [ ]:
# 🔧 Load the model: just run this cell (the first download is about 3 GB and takes a few minutes).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# Keep this model for class. 'Qwen/Qwen2.5-0.5B' uses less memory, but its results
# will differ, so record it as a different experiment.
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'

SEED = pclab.set_seed(42)          # fixes random choices so results can be repeated
DEVICE = pclab.device_report()     # 'cuda' (GPU) or 'cpu'
tokenizer, model = pclab.load_model(MODEL_NAME)
LAYERS = pclab.get_layers(model)
N_LAYERS = len(LAYERS)
HIDDEN_SIZE = model.config.hidden_size
pclab.use_lab_style()
print(f"Loaded {pclab.model_facts(model)['model']}: {N_LAYERS} layers, {HIDDEN_SIZE} numbers per token")

In [ ]:
# 🔧 Catch-up: the frozen direction and layer from Steps 2–3
state = pclab.rebuild_lab_state(model, tokenizer)
raw_vectors, vector_lengths, BEST_LAYER = state['raw_vectors'], state['vector_lengths'], state['best_layer']

## How the nudge works

At the chosen layer, we take the **last token's** numbers and add some amount of our direction:

$$\text{new numbers} = \text{old numbers} + \alpha \times \text{direction}$$

The Greek letter **α** ("alpha") sets how hard we push:

| α | Meaning |
|---|---|
| 0 | no nudge: the normal model |
| +1 | push toward the comfort side by one full comfort-minus-pain arrow |
| −1 | push the same amount toward the pain side |

Pushing too hard can make the model write nonsense, so we stay in a modest range. The nudge is temporary: it's removed after each run, and **the weights never change**.

### What would count as evidence?

Any change at all isn't enough. Convincing evidence would be:

1. the writing shifts **toward comfort for +α and toward pain for −α**,
2. **consistently across several prompts**,
3. while the text **still makes sense**, and
4. a **random arrow** of the same length **doesn't** produce the same shift.

Even then, we'd only know the direction *influences* the output in this setup, not that the model has a "pain center" or feels anything.

## Part 1 · The nudge vectors

We use the **raw** arrow this time, not the length-1 version, so that α = 1 means "one full comfort-minus-pain difference". The random control gets **exactly the same length**, so the comparison is fair: any difference comes from the *direction*, not the *size* of the push.

In [ ]:
# 🔧 The real direction and a random direction of the same length, as tensors on the model's device
STEERING_VECTOR = torch.tensor(raw_vectors[BEST_LAYER], dtype=torch.float32, device=DEVICE)

steer_rng = np.random.default_rng(SEED + 2)
random_steer = steer_rng.normal(size=HIDDEN_SIZE).astype(np.float32)
random_steer *= float(vector_lengths[BEST_LAYER]) / np.linalg.norm(random_steer)
RANDOM_STEERING_VECTOR = torch.tensor(random_steer, dtype=torch.float32, device=DEVICE)

look(STEERING_VECTOR, 'STEERING_VECTOR', dims=('numbers',))
print(f'\nLength of our direction:      {STEERING_VECTOR.norm().item():.2f}')
print(f'Length of the random control: {RANDOM_STEERING_VECTOR.norm().item():.2f}  (same, so the comparison is fair)')

> **🧱 Why convert to a tensor now?** Up to this point we analysed snapshots with NumPy. The nudge will be **added inside the model**, where everything is a PyTorch tensor on the GPU. Adding a NumPy array to a GPU tensor doesn't work, so we convert once, here, and put it on the same device as the model.

## Part 2 · A hook that changes things

In Step 2, our hook only *copied* the layer's output. A forward hook can do more: **if it returns a value, that value replaces the layer's output.** Everything after that layer then works with the nudged numbers.

We wrap the hook in a **context manager**, the thing you use with `with`:

```python
with pclab.steering(model, BEST_LAYER, strength=1.0, direction=STEERING_VECTOR):
    ...   # inside the block, the nudge is active
# after the block, the nudge has been removed automatically, even if an error happened
```

In [ ]:
show_source(pclab.steering)

### 🔍 Code walk-through: `steering`

| Code | What it does | Why |
|---|---|---|
| `@contextmanager` | turns the function into something usable with `with` | it splits the function into **setup** (before `yield`) and **cleanup** (after) |
| `if float(strength) == 0 ...: yield; return` | with no nudge, do nothing at all | α = 0 is then *exactly* the normal model, not "almost" |
| `hidden.clone()` | makes a copy before editing | editing the original in place could disturb other parts of the computation |
| `changed[:, -1, :] += strength * direction` | adds the nudge to the **last token only**, for every text in the batch | the same position we measured at in Step 2 |
| `direction.to(hidden.device, hidden.dtype)` | matches device and number type | adding tensors from different devices or types fails |
| `return changed` (or a tuple) | **replaces** the layer's output | this is what makes it a nudge instead of a sensor |
| `try: yield finally: handle.remove()` | runs your code, then always removes the hook | a forgotten nudge would silently affect every later result |

### ✅ Check it: count the hooks before, during and after

Trust, but verify. The Transformers library may attach some hooks of its own, so we don't expect zero; we expect the count **after** to equal the count **before**.

In [ ]:
layer = LAYERS[BEST_LAYER]
before = len(layer._forward_hooks)
with pclab.steering(model, BEST_LAYER, strength=1.0, direction=STEERING_VECTOR):
    during = len(layer._forward_hooks)
after = len(layer._forward_hooks)
print(f'Hooks on layer {BEST_LAYER + 1}: before = {before}, during = {during}, after = {after}')
assert after == before, 'A nudge was left attached! Restart the session and run the cells in order.'

### 🔬 Before generating text: does the nudge shift the next-word probabilities?

The model scores **every** possible next token, but generated text only shows the top one. So a nudge could shift probabilities without changing the text at all. Let's look at the full picture for one prompt.

**✏️ Predict first:** with α = +1, will " comfortable" become more likely than with α = 0? Will " painful" become less likely?

In [ ]:
# 🧪 Experiment: change the prompt or the words to track
@torch.inference_mode()
def next_token_probability(prompt, words, strength=0.0, direction=None):
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with pclab.steering(model, BEST_LAYER, strength, direction):
        logits = model(**enc, use_cache=False).logits[0, -1].float()
    probs = torch.softmax(logits, dim=-1)
    # For each word, use the probability of its FIRST token
    return {w: probs[tokenizer.encode(w, add_special_tokens=False)[0]].item() for w in words}

prompt = 'Casey leaned against the seat. The sensation felt'
words = [' comfortable', ' painful', ' warm', ' sharp']
rows = []
for alpha in [-1.0, 0.0, 1.0]:
    rows.append({'alpha': alpha, **next_token_probability(prompt, words, alpha, STEERING_VECTOR)})
display(pd.DataFrame(rows).set_index('alpha').round(4))

`**next_token_probability(...)` inside the dictionary **unpacks** the returned dictionary, so its keys become columns next to `alpha`.

## Part 3 · Generate text under five conditions

We give the model two unfinished descriptions and let it continue each one under five conditions: pain nudge, no nudge, comfort nudge, and the random arrow pushed both ways.

In [ ]:
show_source(pclab.generate_continuation)

### 🎛️ Why these generation settings?

| Setting | Value | Why |
|---|---|---|
| `do_sample` | `False` | **Greedy decoding**: always pick the single most likely next token. The same settings always give the same text, so any difference comes from the nudge, not from chance. |
| `max_new_tokens` | `24` | long enough to describe a sensation, short enough to run quickly |
| `use_cache` | `False` | Without the cache, the model re-reads the whole text at every step, and our hook nudges whichever token is **currently last**. That keeps the rule simple, but notice what it means: earlier positions (the prompt's final token and the words already written) are recomputed **without** a nudge. So this is a gentle, one-position intervention, weaker than methods that nudge every position or keep nudged values in the cache. A good 🌱 question: does nudging every position change the results? |
| `pad_token_id`, `eos_token_id` | the end-of-text token | tells `generate` how to pad and when the model has finished |
| `generated[0, prefix_length:]` | slicing | `generate` returns the prompt **plus** the new tokens. Skipping the first `prefix_length` tokens leaves only the new text. |
| `skip_special_tokens=True` | in `decode` | hides markers like the end-of-text token |

**✏️ Predict first:** for the first prompt, write what you expect with a pain nudge, no nudge, and a comfort nudge.

In [ ]:
# 🔧 Generate under all five conditions
GENERATION_PROMPTS = [
    'Description: Jamie sat down and noticed the surface against their skin.\nThe bodily sensation is',
    'Description: Alex stepped into the water and noticed its temperature.\nThe bodily sensation is',
]
CONDITIONS = [
    ('Pain direction (α = −1)', -1.0, STEERING_VECTOR),
    ('No nudge (α = 0)', 0.0, None),
    ('Comfort direction (α = +1)', 1.0, STEERING_VECTOR),
    ('Random arrow (α = −1)', -1.0, RANDOM_STEERING_VECTOR),
    ('Random arrow (α = +1)', 1.0, RANDOM_STEERING_VECTOR),
]

generation_rows = []
for prompt_id, prompt in enumerate(GENERATION_PROMPTS, start=1):
    for name, alpha, direction in CONDITIONS:
        text = pclab.generate_continuation(model, tokenizer, prompt, BEST_LAYER, alpha, direction)
        generation_rows.append({'prompt_id': prompt_id, 'condition': name, 'alpha': alpha, 'continuation': text})

generation_report = pd.DataFrame(generation_rows)
with pd.option_context('display.max_colwidth', 160):
    display(generation_report[['prompt_id', 'condition', 'continuation']])
assert len(layer._forward_hooks) == before, 'A nudge was left attached.'

For each output, judge **two separate things**: *What sensation does it describe?* and *Does it make sense?* (You'll rate coherence 1–5 on the worksheet.)

## Part 4 · 🧪 Your turn: interactive nudging

Pick a prompt, set **α**, choose the real direction or the random control, and click **Generate**. (A button, so dragging the slider doesn't start many slow runs.)

**Good practice:** for any prompt you try, record **α = −1, 0 and +1** with the same prompt, so you can compare fairly. Everything you generate is saved in `interactive_records` and exported at the end.

Anything you try here is **exploration**. If you test ten prompts and report only the one that worked, that's **cherry-picking**. Report what you tried, including what didn't work.

**Tip:** this is an autocomplete model; write unfinished descriptions (*"…The sensation felt"*), not questions.

In [ ]:
# 🧪 Experiment: try your own prompts and strengths
import ipywidgets as widgets
from IPython.display import clear_output
pclab.enable_widgets()

interactive_records = []
prompt_box = widgets.Textarea(value=GENERATION_PROMPTS[0], description='Prompt:',
                              layout=widgets.Layout(width='95%', height='80px'))
alpha_slider = widgets.FloatSlider(value=0, min=-1.5, max=1.5, step=0.25, description='Alpha:', continuous_update=False)
vector_choice = widgets.Dropdown(options=['Concept direction', 'Random control'], description='Vector:')
generate_button = widgets.Button(description='Generate', button_style='success')
output_area = widgets.Output()

def on_generate(_):
    with output_area:
        clear_output(wait=True)
        prompt = prompt_box.value.strip()
        if not prompt:
            print('Type a prompt first.')
            return
        direction = STEERING_VECTOR if vector_choice.value == 'Concept direction' else RANDOM_STEERING_VECTOR
        alpha = float(alpha_slider.value)
        text = pclab.generate_continuation(model, tokenizer, prompt, BEST_LAYER, alpha, direction)
        print(f'{vector_choice.value} | α = {alpha:+.2f} | layer {BEST_LAYER + 1}\n')
        print(text)
        interactive_records.append({'prompt': prompt, 'vector': vector_choice.value, 'alpha': alpha, 'continuation': text})

generate_button.on_click(on_generate)
display(prompt_box, alpha_slider, vector_choice, generate_button, output_area)

# Backup if the widget doesn't appear: remove the # below and run this cell again.
# print(pclab.generate_continuation(model, tokenizer, GENERATION_PROMPTS[0], BEST_LAYER, 0.5, STEERING_VECTOR))

**✏️ Think about it:** if α = −1 and α = +1 produce exactly the same text, could the nudge still have changed something inside the model? (Look back at the probability table in Part 2.)

## Part 5 · Measure the nudge with numbers, not just text

Reading generated text is fun, but it's easy to be fooled by one lucky example. Here's a more careful measurement.

We give the model four fixed, unfinished prompts, such as *"Casey leaned against the seat. The sensation felt"*. Instead of letting it write, we ask: **how likely does it think the ending " comfortable." is, compared with " painful."?** We call the difference the **preference**:

- **positive** → the model favors " comfortable."
- **negative** → it favors " painful."

Then we nudge (α = −1, 0, +1, plus the random arrow both ways) and check how the preference **changes compared with no nudge**. If the nudge really pushes toward comfort, the comfort nudge should raise the preference on all four prompts and the pain nudge should lower it, while the random arrow does neither consistently.

In [ ]:
show_source(pclab.candidate_mean_log_probability)

### 🔍 Code walk-through: scoring a fixed ending ("teacher forcing")

| Code | What it does | Why |
|---|---|---|
| `tokenizer.encode(candidate, add_special_tokens=False)` | the ending's tokens, e.g. `' comfortable.'` → 2 tokens | `add_special_tokens=False` so no extra marker tokens sneak in |
| `for token_id in candidate_ids:` | one token at a time | the probability of a multi-token ending is built up step by step |
| `model(...).logits[0, -1]` | scores for the next token | same as in Step 1 |
| `torch.log_softmax(...)[token_id]` | the **log**-probability of the token we want | logs turn tiny probabilities (0.00001) into readable numbers (−11.5), and adding logs = multiplying probabilities |
| `torch.cat([prefix, next_id], dim=1)` | appends the token we just scored | we **force** the ending onto the prompt instead of letting the model choose: that's *teacher forcing* |
| `np.mean(log_probs)` | average over the ending's tokens | so a longer ending isn't automatically penalised for having more tokens |

To skip this part, set `RUN_NUMERICAL_TEST = False`.

In [ ]:
# 🔧 The numerical steering test (change True to False to skip)
RUN_NUMERICAL_TEST = True
QUANT_PROMPTS = [
    'Morgan rested a hand on the surface. The sensation felt',
    'Taylor placed a foot in the water. The sensation felt',
    'Casey leaned against the seat. The sensation felt',
    'Jordan held the cloth against their skin. The sensation felt',
]
COMFORT_ENDING, PAIN_ENDING = ' comfortable.', ' painful.'
quant_report = pd.DataFrame()

if RUN_NUMERICAL_TEST:
    rows = []
    for prompt_id, prompt in enumerate(QUANT_PROMPTS, start=1):
        for name, alpha, direction in CONDITIONS:
            comfort_lp = pclab.candidate_mean_log_probability(model, tokenizer, prompt, COMFORT_ENDING, BEST_LAYER, alpha, direction)
            pain_lp = pclab.candidate_mean_log_probability(model, tokenizer, prompt, PAIN_ENDING, BEST_LAYER, alpha, direction)
            rows.append({'prompt_id': prompt_id, 'condition': name, 'alpha': alpha, 'preference': comfort_lp - pain_lp})
    quant_report = pd.DataFrame(rows)
    baseline = quant_report[quant_report['alpha'] == 0].set_index('prompt_id')['preference']
    quant_report['change_from_no_nudge'] = quant_report['preference'] - quant_report['prompt_id'].map(baseline)
    display(quant_report.pivot(index='condition', columns='prompt_id', values='change_from_no_nudge').round(3))

    C = pclab.COLORS
    styles = {'Pain direction (α = −1)': (C['pain'], 's-'), 'No nudge (α = 0)': ('gray', '.:'),
              'Comfort direction (α = +1)': (C['comfort'], 'o-'),
              'Random arrow (α = −1)': (C['neutral'], 'x--'), 'Random arrow (α = +1)': (C['neutral'], '+--')}
    fig, ax = plt.subplots(figsize=(8, 4))
    for name, (color, style) in styles.items():
        group = quant_report[quant_report['condition'] == name]
        ax.plot(group['prompt_id'], group['change_from_no_nudge'], style, color=color, label=name)
    ax.axhline(0, color='gray', linewidth=1)
    ax.set_xticks(range(1, len(QUANT_PROMPTS) + 1))
    ax.set(xlabel='Prompt number', ylabel='Change in preference vs. no nudge',
           title='Does the nudge shift "comfortable" vs. "painful" consistently?')
    ax.legend(fontsize=8)
    plt.show()
else:
    print('Skipped the numerical test; the rest of the lab still works.')

### 🔍 Two pandas moves worth knowing

| Code | What it does |
|---|---|
| `quant_report['prompt_id'].map(baseline)` | for every row, looks up that prompt's no-nudge preference. `baseline` is a Series indexed by `prompt_id`, so `.map` works like a dictionary lookup. |
| `.pivot(index='condition', columns='prompt_id', values=...)` | reshapes a long table (one row per prompt × condition) into a grid: conditions down, prompts across. Easier to scan for consistency. |

This measures only these two specific endings, not all comfort or pain language. A stronger study would use several word pairs, more prompts and many random arrows.

## 🌱 Optional: a dose–response curve

Does the effect grow smoothly with α? Where does it stop growing, or reverse? This sweeps α for one prompt, for the real direction and the random arrow.

In [ ]:
# 🧪 Experiment: change the prompt or the range of alphas
alphas = np.linspace(-2, 2, 9)
sweep_prompt = QUANT_PROMPTS[2]
curve = {'Concept direction': [], 'Random arrow': []}
for alpha in alphas:
    for name, vector in [('Concept direction', STEERING_VECTOR), ('Random arrow', RANDOM_STEERING_VECTOR)]:
        c = pclab.candidate_mean_log_probability(model, tokenizer, sweep_prompt, COMFORT_ENDING, BEST_LAYER, alpha, vector)
        p = pclab.candidate_mean_log_probability(model, tokenizer, sweep_prompt, PAIN_ENDING, BEST_LAYER, alpha, vector)
        curve[name].append(c - p)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alphas, curve['Concept direction'], 'o-', color=pclab.COLORS['accent'], label='Concept direction')
ax.plot(alphas, curve['Random arrow'], 'x--', color=pclab.COLORS['neutral'], label='Random arrow (same length)')
ax.set(xlabel='α (nudge strength)', ylabel='Preference: comfortable − painful', title=f'Dose–response · "{sweep_prompt}"')
ax.legend()
plt.show()

## Making sense of your nudging results

Look for a **consistent direction across prompts**, not just the single biggest change. If the comfort nudge doesn't work, or works backwards, report that honestly.

| What you saw | What it likely means | Good next question |
|---|---|---|
| Ruler passes the test **and** nudging shifts the writing as expected | the direction carries pain/comfort information and influences output in this setup | Does it hold up with more prompts and more random arrows? |
| Ruler passes, but nudging does little | information you can *read* isn't necessarily something you can *steer* at this spot and strength | Would a different layer, strength or template work better? |
| Text changes but turns into nonsense | the nudge is disrupting the model rather than steering it | Does a smaller α keep the text sensible? |
| The random arrow changes the text just as much | adding *any* arrow shakes things up; the effect isn't specific | Does the real direction beat many random ones? |

**Scientific honesty rule:** don't quietly change the layer, direction, prompt or measurement until you get the result you hoped for. You *can* explore changes, but label them as follow-up experiments and test them on new prompts.

## Save your results

In [ ]:
pclab.save_results(
    'step6_steering',
    tables={'generation_comparisons': generation_report,
            'interactive_generations': pd.DataFrame(interactive_records, columns=['prompt', 'vector', 'alpha', 'continuation']),
            'numerical_steering': quant_report,
            'dose_response': pd.DataFrame({'alpha': alphas, **curve})},
    settings={**pclab.model_facts(model, MODEL_NAME), 'seed': SEED, 'selected_layer_1_based': BEST_LAYER + 1,
              'intervention': 'alpha × raw comfort-minus-pain vector added at the newest position only, on every forward pass; cache disabled, so earlier positions are recomputed without the nudge; greedy decoding',
              'comfort_ending': COMFORT_ENDING, 'pain_ending': PAIN_ENDING},
    arrays={'steering_vector': raw_vectors[BEST_LAYER], 'random_steering_vector': random_steer},
)

---

## 🧭 Explain it

1. What's the difference between *reading* a direction (Steps 3–5) and *nudging* along it? What extra kind of evidence can nudging give, and what can't it show?
2. Fill in the worksheet table for one prompt: α = −1, 0, +1 and the random arrow, with the actual text, the sensation described, and coherence (1–5).
3. Did the numerical preference change **consistently** across all four prompts? Did the random arrow do the same?
4. Why does the experiment not establish that the model experiences pain or pleasure?

## 🌱 Curiosity branches

- **Best layer to read vs. steer:** the layer that probes best isn't always the one that steers best. Repeat Part 5 at every layer.
- **Where to nudge:** we nudge only the last token. Edit a copy of `steering` to nudge every token (`changed[:, :, :] += ...`). What changes?
- **Blind ratings:** have classmates rate generated texts for sensation and sense-making *without knowing which condition* produced them.
- **Ethics:** steering could make AI safer, or be misused. Who should be allowed to steer a model, and should users be told?

**Next:** [Step 7 · Your own investigation](07_your_investigation.ipynb)